# PROTECT-AI — Zero-shot 3RS rupture prediction with Qwen2.5-VL-3B-Instruct

This notebook is a **first visual-only zero-shot experiment**.

Pipeline:

1. Split a psychotherapy recording into exact 60-second segments.
2. Extract simple interpretable video statistics from each segment.
3. Send each 1-minute segment plus the extracted feature summary to **Qwen/Qwen2.5-VL-3B-Instruct**.
4. Ask the VLM to independently estimate four 3RS v2022 rupture-salience targets:
   - `WD_P` — patient withdrawal
   - `WD_T` — therapist withdrawal
   - `CF_P` — patient confrontation
   - `CF_T` — therapist confrontation
5. Save structured predictions, evidence, confidence and feature values.
6. Optionally compare zero-shot predictions with human 3RS labels.

### Important scope

Qwen2.5-VL can process the **video stream**, but this experiment does **not** use the spoken audio or transcript. Therefore, it must not infer verbal content such as criticism, short answers, disagreement, or topic changes unless the behavior is visibly supported. This notebook should be treated as an exploratory baseline, not as a clinical decision system.

All inference is local when the model is downloaded locally; the therapy video is not intentionally sent to an external API.


## 1. Install dependencies

You also need **FFmpeg** available from the command line.

The plain `qwen-vl-utils` installation is kept portable for Windows/Linux.


In [1]:
# %pip install -U "transformers>=4.49,<6" accelerate qwen-vl-utils opencv-python-headless pandas numpy tqdm scikit-learn


## 2. Configuration

`ROLE_DESCRIPTION` is important. Change it to match the actual camera layout.

Examples:

- `"Patient is the person on the LEFT; therapist is the person on the RIGHT."`
- `"Only the patient is visible. Therapist is off camera."`

The engineered left/right features follow `PATIENT_SIDE` and `THERAPIST_SIDE`.


In [2]:
from pathlib import Path
import json
import math
import re
import shutil
import subprocess
import time

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

VIDEO_PATH = Path(r"C:\Data\Sequence_model\Memopsy_videos\CONVERTED\PSYCH-DZPG_Klambt-N\401033\401033_S1.mp4")

OUTPUT_DIR = Path("./qwen_3rs_zero_shot_output")
SEGMENT_DIR = OUTPUT_DIR / "segments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEGMENT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

SEGMENT_SECONDS = 60
VLM_VIDEO_FPS = 0.5
VLM_HEIGHT = 280
VLM_WIDTH = 448

FEATURE_FPS = 2.0
FEATURE_RESIZE_WIDTH = 640

PATIENT_SIDE = "left"
THERAPIST_SIDE = "right"
ROLE_DESCRIPTION = "Patient is the person on the LEFT; therapist is the person on the RIGHT."

MAX_NEW_TOKENS = 550
MAX_SEGMENTS = None

HUMAN_LABELS_CSV = None
BINARY_RUPTURE_THRESHOLD = 2.0

assert PATIENT_SIDE in {"left", "right"}
assert THERAPIST_SIDE in {"left", "right"}
assert PATIENT_SIDE != THERAPIST_SIDE


## 3. 3RS v2022 zero-shot rubric used in this experiment

This compact rubric is based on the project/manual guidance used for the current 1-minute annotation setup.

### Construct

An alliance rupture is a strain, tension or breakdown in collaboration or connection. The revised 3RS treats rupture as a **dyadic** process: both patient and therapist can show withdrawal or confrontation.

### Withdrawal — moving away

- `WD_P`: patient moves away from the therapist, emotional contact, or therapeutic work.
- `WD_T`: therapist moves away from the patient, difficult material, or relational strain.
- Potential markers include avoidance, emotional shutdown, reduced engagement, distancing, deference, or disengagement.

### Confrontation — moving against

- `CF_P`: patient moves against the therapist or therapy.
- `CF_T`: therapist moves against the patient.
- Potential markers include rejection, criticism, pressure, hostility, rigid control, defensiveness, or visibly forceful disagreement.

### Salience scale

- `1` = not salient
- `2` = between not salient and somewhat salient
- `3` = somewhat salient
- `4` = between somewhat and very salient
- `5` = very salient

### Visual-only rule for this baseline

Some 3RS markers are mainly verbal. The VLM must **not** invent words, intentions, diagnoses, or unseen events. It should use visible evidence such as posture, orientation, gaze direction, facial behavior, gesture, physical distancing, visible disengagement, visible tension, or forceful interaction. Low motion or gaze aversion alone is not automatically a rupture.


In [3]:
THREE_RS_RUBRIC = """
You are rating one 60-second psychotherapy segment using a visual-only adaptation of the
Rupture Resolution Rating System (3RS v2022).

A rupture is a strain, tension, or breakdown in collaboration or connection.

Rate FOUR separate constructs:

WD_P = patient withdrawal:
The patient visibly moves away from the therapist, emotional contact, or therapeutic work.
Possible visual signs may include disengagement, distancing, shutting down, avoiding contact,
markedly reduced responsiveness, or withdrawn posture. These signs are not sufficient by
themselves; judge the interaction in context.

WD_T = therapist withdrawal:
The therapist visibly moves away from the patient, difficult material, or relational strain.
Possible visual signs may include distancing, disengagement, avoidance of the patient's
visible concern, or reduced relational responsiveness.

CF_P = patient confrontation:
The patient visibly moves against the therapist or therapeutic work.
Possible visual signs may include forceful rejection, hostile or pressuring gestures,
visible criticism/disagreement, controlling behavior, or marked interpersonal tension.

CF_T = therapist confrontation:
The therapist visibly moves against the patient.
Possible visual signs may include pressure, criticism, controlling gestures,
defensive behavior, rigid direction, or marked interpersonal tension.

Use the 1-5 salience scale:
1 = not salient
2 = between not salient and somewhat salient
3 = somewhat salient
4 = between somewhat and very salient
5 = very salient

VISUAL-ONLY RESTRICTIONS:
- Do not claim to know what either person said.
- Do not infer criticism, agreement, topic shifts, short answers, or other verbal markers from lip movement.
- Do not infer diagnosis, emotion, intention, or alliance quality from a single facial expression.
- Do not treat gaze aversion, stillness, smiling, or movement as a rupture by itself.
- Use the full temporal interaction and multiple visible cues.
- If the relevant person is not visible enough, keep confidence low and state the limitation.
- The supplied numeric video features are auxiliary measurements, not rupture labels.
""".strip()


## 4. Verify FFmpeg and inspect the source video

In [4]:
def require_command(name: str):
    path = shutil.which(name)
    if path is None:
        raise RuntimeError(
            f"{name} was not found on PATH. Install FFmpeg and make sure both ffmpeg and ffprobe are available."
        )
    return path

require_command("ffmpeg")
require_command("ffprobe")

if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Set VIDEO_PATH to your input video. Current path: {VIDEO_PATH}")

def get_video_duration(video_path: Path) -> float:
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(video_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return float(result.stdout.strip())

duration_sec = get_video_duration(VIDEO_PATH)
print(f"Video: {VIDEO_PATH}")
print(f"Duration: {duration_sec / 60:.2f} min")
print(f"Expected segments: {math.ceil(duration_sec / SEGMENT_SECONDS)}")


Video: C:\Data\Sequence_model\Memopsy_videos\CONVERTED\PSYCH-DZPG_Klambt-N\401033\401033_S1.mp4
Duration: 43.39 min
Expected segments: 44


## 5. Split the recording into exact 1-minute segments

The video is re-encoded so that segment boundaries are exact rather than being restricted to existing keyframes.


In [5]:
def split_video_into_segments(
    video_path: Path,
    out_dir: Path,
    segment_seconds: int = 60,
) -> pd.DataFrame:
    duration = get_video_duration(video_path)
    n_segments = math.ceil(duration / segment_seconds)
    rows = []

    for idx in tqdm(range(n_segments), desc="Creating 1-minute segments"):
        start = idx * segment_seconds
        length = min(segment_seconds, duration - start)
        out_path = out_dir / f"{video_path.stem}_seg_{idx:04d}_{start:06.0f}-{start+length:06.0f}s.mp4"

        if not out_path.exists():
            cmd = [
                "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                "-ss", str(start),
                "-i", str(video_path),
                "-t", str(length),
                "-map", "0:v:0",
                "-map", "0:a?",
                "-c:v", "libx264",
                "-preset", "veryfast",
                "-crf", "18",
                "-c:a", "aac",
                "-b:a", "128k",
                "-movflags", "+faststart",
                str(out_path),
            ]
            subprocess.run(cmd, check=True)

        rows.append({
            "segment_idx": idx,
            "start_sec": start,
            "end_sec": start + length,
            "duration_sec": length,
            "segment_path": str(out_path.resolve()),
        })

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "segments_manifest.csv", index=False)
    return df

segments_df = split_video_into_segments(VIDEO_PATH, SEGMENT_DIR, SEGMENT_SECONDS)
segments_df.head()


Creating 1-minute segments:   0%|          | 0/44 [00:00<?, ?it/s]

,segment_idx,start_sec,end_sec,duration_sec,segment_path
0,0,0,60.0,60.0,C:\Data\Sequence_model\facs-openface-tools\qwe...
1,1,60,120.0,60.0,C:\Data\Sequence_model\facs-openface-tools\qwe...
2,2,120,180.0,60.0,C:\Data\Sequence_model\facs-openface-tools\qwe...
3,3,180,240.0,60.0,C:\Data\Sequence_model\facs-openface-tools\qwe...
4,4,240,300.0,60.0,C:\Data\Sequence_model\facs-openface-tools\qwe...


## 6. Extract simple video features

These are intentionally simple and interpretable. They do **not** attempt to diagnose emotion or directly classify rupture.

Features include:

- frame brightness and blur/visibility proxy
- optical-flow motion for the whole frame and each side
- face visibility fraction using OpenCV's built-in frontal-face detector
- mean visible face area on each side
- face-center movement/jitter on each side

The VLM receives these values only as auxiliary context.


In [6]:
import cv2
import sys

print("Python:", sys.executable)
print("cv2:", cv2)
print("cv2 path:", getattr(cv2, "__file__", None))
print("cv2 version:", getattr(cv2, "__version__", None))
print("CascadeClassifier available:", hasattr(cv2, "CascadeClassifier"))

Python: c:\Data\Sequence_model\facs-openface-tools\.venv\Scripts\python.exe
cv2: <module 'cv2' from 'c:\\Data\\Sequence_model\\facs-openface-tools\\.venv\\Lib\\site-packages\\cv2\\__init__.py'>
cv2 path: c:\Data\Sequence_model\facs-openface-tools\.venv\Lib\site-packages\cv2\__init__.py
cv2 version: 4.12.0
CascadeClassifier available: True


In [7]:
# FACE_CASCADE = cv2.CascadeClassifier(
#     cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
# )

# def _resize_keep_aspect(frame, width):
#     h, w = frame.shape[:2]
#     if w <= width:
#         return frame
#     new_h = int(h * width / w)
#     return cv2.resize(frame, (width, new_h), interpolation=cv2.INTER_AREA)

# def _side_from_x(x_center, width):
#     return "left" if x_center < width / 2 else "right"

# def extract_video_features(
#     video_path: Path,
#     sample_fps: float = 2.0,
#     resize_width: int = 640,
# ) -> dict:
#     cap = cv2.VideoCapture(str(video_path))
#     if not cap.isOpened():
#         raise RuntimeError(f"Could not open {video_path}")

#     source_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
#     frame_step = max(1, int(round(source_fps / sample_fps)))

#     brightness = []
#     blur = []
#     motion_all = []
#     motion_left = []
#     motion_right = []

#     side_visible = {"left": [], "right": []}
#     side_area = {"left": [], "right": []}
#     side_centers = {"left": [], "right": []}

#     prev_gray = None
#     frame_idx = 0
#     sampled = 0

#     while True:
#         ok, frame = cap.read()
#         if not ok:
#             break

#         if frame_idx % frame_step != 0:
#             frame_idx += 1
#             continue

#         frame = _resize_keep_aspect(frame, resize_width)
#         gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#         h, w = gray.shape

#         brightness.append(float(gray.mean()))
#         blur.append(float(cv2.Laplacian(gray, cv2.CV_64F).var()))

#         if prev_gray is not None and prev_gray.shape == gray.shape:
#             flow = cv2.calcOpticalFlowFarneback(
#                 prev_gray, gray, None,
#                 0.5, 3, 15, 3, 5, 1.2, 0
#             )
#             mag = np.linalg.norm(flow, axis=2)
#             mid = w // 2
#             motion_all.append(float(np.mean(mag)))
#             motion_left.append(float(np.mean(mag[:, :mid])))
#             motion_right.append(float(np.mean(mag[:, mid:])))
#         prev_gray = gray

#         faces = FACE_CASCADE.detectMultiScale(
#             gray,
#             scaleFactor=1.1,
#             minNeighbors=5,
#             minSize=(40, 40),
#         )

#         current = {"left": [], "right": []}
#         for x, y, fw, fh in faces:
#             cx = x + fw / 2
#             cy = y + fh / 2
#             side = _side_from_x(cx, w)
#             current[side].append((x, y, fw, fh, cx, cy))

#         for side in ("left", "right"):
#             if current[side]:
#                 best = max(current[side], key=lambda b: b[2] * b[3])
#                 x, y, fw, fh, cx, cy = best
#                 side_visible[side].append(1.0)
#                 side_area[side].append(float((fw * fh) / (w * h)))
#                 side_centers[side].append((float(cx / w), float(cy / h)))
#             else:
#                 side_visible[side].append(0.0)
#                 side_area[side].append(0.0)
#                 side_centers[side].append(None)

#         sampled += 1
#         frame_idx += 1

#     cap.release()

#     def safe_mean(values):
#         return float(np.mean(values)) if values else None

#     def center_jitter(centers):
#         valid = np.array([p for p in centers if p is not None], dtype=np.float32)
#         if len(valid) < 2:
#             return None
#         diffs = np.linalg.norm(np.diff(valid, axis=0), axis=1)
#         return float(np.mean(diffs))

#     features = {
#         "sampled_frames": sampled,
#         "brightness_mean_0_255": safe_mean(brightness),
#         "blur_laplacian_var_mean": safe_mean(blur),
#         "motion_mean": safe_mean(motion_all),
#         "left_motion_mean": safe_mean(motion_left),
#         "right_motion_mean": safe_mean(motion_right),
#         "left_face_visible_fraction": safe_mean(side_visible["left"]),
#         "right_face_visible_fraction": safe_mean(side_visible["right"]),
#         "left_face_area_fraction_mean": safe_mean(side_area["left"]),
#         "right_face_area_fraction_mean": safe_mean(side_area["right"]),
#         "left_face_center_jitter": center_jitter(side_centers["left"]),
#         "right_face_center_jitter": center_jitter(side_centers["right"]),
#     }

#     p = PATIENT_SIDE
#     t = THERAPIST_SIDE
#     features.update({
#         "patient_motion_mean": features[f"{p}_motion_mean"],
#         "therapist_motion_mean": features[f"{t}_motion_mean"],
#         "patient_face_visible_fraction": features[f"{p}_face_visible_fraction"],
#         "therapist_face_visible_fraction": features[f"{t}_face_visible_fraction"],
#         "patient_face_area_fraction_mean": features[f"{p}_face_area_fraction_mean"],
#         "therapist_face_area_fraction_mean": features[f"{t}_face_area_fraction_mean"],
#         "patient_face_center_jitter": features[f"{p}_face_center_jitter"],
#         "therapist_face_center_jitter": features[f"{t}_face_center_jitter"],
#     })

#     return features

# feature_rows = []
# for row in tqdm(segments_df.itertuples(index=False), total=len(segments_df), desc="Extracting video features"):
#     feat = extract_video_features(Path(row.segment_path), FEATURE_FPS, FEATURE_RESIZE_WIDTH)
#     feature_rows.append({"segment_idx": row.segment_idx, **feat})

# features_df = pd.DataFrame(feature_rows)
# features_df.to_csv(OUTPUT_DIR / "video_features.csv", index=False)
# features_df.head()


In [8]:
FEATURES_PATH = OUTPUT_DIR / "video_features.csv"

# Load previous work after a kernel restart
if FEATURES_PATH.exists():
    features_df = pd.read_csv(FEATURES_PATH)
    print(f"Loaded {len(features_df)} previously extracted segments.")
else:
    features_df = pd.DataFrame()

done_segments = (
    set(features_df["segment_idx"].astype(int))
    if "segment_idx" in features_df.columns
    else set()
)

for row in tqdm(
    segments_df.itertuples(index=False),
    total=len(segments_df),
    desc="Extracting video features",
):
    if row.segment_idx in done_segments:
        continue

    feat = extract_video_features(
        Path(row.segment_path),
        FEATURE_FPS,
        FEATURE_RESIZE_WIDTH,
    )

    new_row = pd.DataFrame([{
        "segment_idx": row.segment_idx,
        **feat,
    }])

    features_df = pd.concat(
        [features_df, new_row],
        ignore_index=True,
    )

    # Save after every segment so progress survives crashes/restarts
    features_df = (
        features_df
        .drop_duplicates("segment_idx", keep="last")
        .sort_values("segment_idx")
        .reset_index(drop=True)
    )

    features_df.to_csv(FEATURES_PATH, index=False)

print(f"Features available for {len(features_df)} segments.")
features_df.head()

Loaded 44 previously extracted segments.


Extracting video features:   0%|          | 0/44 [00:00<?, ?it/s]

Features available for 44 segments.


,segment_idx,sampled_frames,brightness_mean_0_255,blur_laplacian_var_mean,motion_mean,left_motion_mean,right_motion_mean,left_face_visible_fraction,right_face_visible_fraction,left_face_area_fraction_mean,...,left_face_center_jitter,right_face_center_jitter,patient_motion_mean,therapist_motion_mean,patient_face_visible_fraction,therapist_face_visible_fraction,patient_face_area_fraction_mean,therapist_face_area_fraction_mean,patient_face_center_jitter,therapist_face_center_jitter
0,0,120,102.599367,422.900300,2.476088,1.963836,2.988339,0.808333,0.000000,0.011337,...,0.016690,NaN,1.963836,2.988339,0.808333,0.000000,0.011337,0.000000,0.016690,NaN
1,1,120,109.266043,331.629893,2.373708,1.521814,3.225602,0.050000,0.000000,0.001736,...,0.024996,NaN,1.521814,3.225602,0.050000,0.000000,0.001736,0.000000,0.024996,NaN
2,2,120,110.336027,324.874792,1.554309,1.933896,1.174722,0.066667,0.000000,0.001920,...,0.070294,NaN,1.933896,1.174722,0.066667,0.000000,0.001920,0.000000,0.070294,NaN
3,3,120,113.703652,303.837677,1.200124,0.439835,1.960412,0.008333,0.008333,0.000375,...,NaN,NaN,0.439835,1.960412,0.008333,0.008333,0.000375,0.000134,NaN,NaN
4,4,120,110.779574,322.269670,0.749322,0.315177,1.183468,0.066667,0.016667,0.000820,...,0.064083,0.113034,0.315177,1.183468,0.066667,0.016667,0.000820,0.000325,0.064083,0.113034


## 7. Load Qwen2.5-VL-3B-Instruct

The model is loaded locally from Hugging Face. `device_map="auto"` lets Accelerate place the model on available hardware.

For a first run, keep `VLM_VIDEO_FPS` around `0.5`–`1.0` to control visual-token usage.


In [9]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        model_dtype = torch.bfloat16
    else:
        model_dtype = torch.float16
else:
    model_dtype = torch.float32

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

model.eval()

print("Model loaded")
print("Device:", next(model.parameters()).device)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model loaded
Device: cuda:0


## 8. Build the structured zero-shot prompt

The model returns both ordinal scores and a zero-shot primary class.

`primary_class` is one of `none`, `WD_P`, `WD_T`, `CF_P`, `CF_T`, or `mixed`.

The four 1–5 scores remain the main outputs. The class is a convenient secondary summary.


In [10]:
def compact_features(features: dict) -> dict:
    keep = [
        "patient_motion_mean",
        "therapist_motion_mean",
        "patient_face_visible_fraction",
        "therapist_face_visible_fraction",
        "patient_face_area_fraction_mean",
        "therapist_face_area_fraction_mean",
        "patient_face_center_jitter",
        "therapist_face_center_jitter",
        "brightness_mean_0_255",
        "blur_laplacian_var_mean",
    ]
    result = {}
    for key in keep:
        value = features.get(key)
        if isinstance(value, (float, np.floating)) and not math.isnan(float(value)):
            result[key] = round(float(value), 4)
        else:
            result[key] = value
    return result

def build_prompt(features: dict) -> str:
    feature_json = json.dumps(compact_features(features), indent=2)

    return f"""
{THREE_RS_RUBRIC}

ROLE LAYOUT:
{ROLE_DESCRIPTION}

AUXILIARY VIDEO FEATURES:
{feature_json}

Interpret the video itself first. Use the numeric features only as weak supporting context.
Do not convert a feature directly into a rupture rating.

Return ONLY valid JSON with this exact schema:

{{
  "scores": {{
    "WD_P": 1,
    "WD_T": 1,
    "CF_P": 1,
    "CF_T": 1
  }},
  "primary_class": "none",
  "confidence": {{
    "WD_P": 0.0,
    "WD_T": 0.0,
    "CF_P": 0.0,
    "CF_T": 0.0
  }},
  "visible_evidence": {{
    "WD_P": [],
    "WD_T": [],
    "CF_P": [],
    "CF_T": []
  }},
  "counterevidence_or_ambiguity": [],
  "visibility_notes": "",
  "segment_summary": ""
}}

Rules for output:
- Every score must be an integer from 1 through 5.
- Every confidence value must be from 0.0 through 1.0.
- Evidence must describe only visible behavior.
- Use "mixed" only when more than one rupture pattern is meaningfully salient.
- If no rupture pattern is visually supported, use "none".
- Keep the segment summary short.
""".strip()


## 9. Run one segment first

This smoke test lets you inspect the prompt, parsing and VRAM use before processing the whole session.


In [11]:
import os

os.environ["FORCE_QWENVL_VIDEO_READER"] = "torchvision"

In [12]:
from qwen_vl_utils import process_vision_info

In [13]:
import os
os.environ["FORCE_QWENVL_VIDEO_READER"] = "torchvision"

import torch

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
)

from qwen_vl_utils import process_vision_info

In [14]:
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())

torch: 2.13.0+cu130
torchvision: 0.28.0+cu130
CUDA: True


In [15]:
import os
os.environ["FORCE_QWENVL_VIDEO_READER"] = "torchvision"

from qwen_vl_utils import process_vision_info

print("Using torchvision backend")

Using torchvision backend


In [16]:
def extract_json_object(text: str) -> dict:
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found in model output:\n{text}")

    return json.loads(text[start:end + 1])

def validate_prediction(pred: dict) -> dict:
    required_scores = ["WD_P", "WD_T", "CF_P", "CF_T"]
    scores = pred.get("scores", {})
    confidence = pred.get("confidence", {})

    for key in required_scores:
        if key not in scores:
            raise ValueError(f"Missing score: {key}")

        scores[key] = int(round(float(scores[key])))
        scores[key] = min(5, max(1, scores[key]))

        c = float(confidence.get(key, 0.0))
        confidence[key] = min(1.0, max(0.0, c))

    allowed = {"none", "WD_P", "WD_T", "CF_P", "CF_T", "mixed"}
    if pred.get("primary_class") not in allowed:
        pred["primary_class"] = "mixed" if max(scores.values()) >= 3 else "none"

    pred["scores"] = scores
    pred["confidence"] = confidence
    return pred

def predict_segment(segment_path: Path, features: dict) -> tuple[dict, str]:
    prompt = build_prompt(features)

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": segment_path.resolve().as_uri(),
                    "fps": VLM_VIDEO_FPS,
                    "resized_height": VLM_HEIGHT,
                    "resized_width": VLM_WIDTH,
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

    chat_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        return_video_kwargs=True,
    )

    inputs = processor(
        text=[chat_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        **video_kwargs,
    )
    inputs = inputs.to(model_device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    raw_text = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    parsed = validate_prediction(extract_json_object(raw_text))
    return parsed, raw_text

test_row = segments_df.iloc[0]
test_features = features_df.loc[
    features_df["segment_idx"] == test_row["segment_idx"]
].iloc[0].drop(labels=["segment_idx"]).to_dict()

test_prediction, test_raw = predict_segment(
    Path(test_row["segment_path"]),
    test_features,
)

print(json.dumps(test_prediction, indent=2, ensure_ascii=False))


qwen-vl-utils using torchcodec to read video.
video_reader_backend torchcodec error, use torchvision as default, msg: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.13.0+cu130) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorc

AttributeError: module 'torchvision.io' has no attribute 'read_video'

## 10. Run the full zero-shot experiment

In [ ]:
def flatten_prediction(segment_row, features, prediction, raw_text, elapsed_sec):
    scores = prediction["scores"]
    conf = prediction["confidence"]

    return {
        "segment_idx": int(segment_row["segment_idx"]),
        "start_sec": float(segment_row["start_sec"]),
        "end_sec": float(segment_row["end_sec"]),
        "segment_path": segment_row["segment_path"],
        "WD_P_pred": scores["WD_P"],
        "WD_T_pred": scores["WD_T"],
        "CF_P_pred": scores["CF_P"],
        "CF_T_pred": scores["CF_T"],
        "primary_class": prediction.get("primary_class"),
        "WD_P_conf": conf["WD_P"],
        "WD_T_conf": conf["WD_T"],
        "CF_P_conf": conf["CF_P"],
        "CF_T_conf": conf["CF_T"],
        "visible_evidence": json.dumps(prediction.get("visible_evidence", {}), ensure_ascii=False),
        "counterevidence_or_ambiguity": json.dumps(
            prediction.get("counterevidence_or_ambiguity", []),
            ensure_ascii=False,
        ),
        "visibility_notes": prediction.get("visibility_notes", ""),
        "segment_summary": prediction.get("segment_summary", ""),
        "inference_sec": elapsed_sec,
        "raw_model_output": raw_text,
        **{f"feat_{k}": v for k, v in compact_features(features).items()},
    }

run_segments = segments_df if MAX_SEGMENTS is None else segments_df.head(MAX_SEGMENTS)

results = []
jsonl_path = OUTPUT_DIR / "qwen_3rs_predictions.jsonl"

with jsonl_path.open("w", encoding="utf-8") as jf:
    for _, segment_row in tqdm(
        run_segments.iterrows(),
        total=len(run_segments),
        desc="Qwen zero-shot 3RS",
    ):
        seg_idx = int(segment_row["segment_idx"])
        features = features_df.loc[
            features_df["segment_idx"] == seg_idx
        ].iloc[0].drop(labels=["segment_idx"]).to_dict()

        started = time.time()

        try:
            prediction, raw_text = predict_segment(
                Path(segment_row["segment_path"]),
                features,
            )
            elapsed = time.time() - started

            record = {
                "segment_idx": seg_idx,
                "start_sec": float(segment_row["start_sec"]),
                "end_sec": float(segment_row["end_sec"]),
                "prediction": prediction,
                "features": compact_features(features),
                "raw_model_output": raw_text,
                "inference_sec": elapsed,
            }

            jf.write(json.dumps(record, ensure_ascii=False) + "\n")
            jf.flush()

            results.append(
                flatten_prediction(
                    segment_row,
                    features,
                    prediction,
                    raw_text,
                    elapsed,
                )
            )

        except Exception as exc:
            elapsed = time.time() - started
            error_record = {
                "segment_idx": seg_idx,
                "start_sec": float(segment_row["start_sec"]),
                "end_sec": float(segment_row["end_sec"]),
                "error": repr(exc),
                "inference_sec": elapsed,
            }
            jf.write(json.dumps(error_record, ensure_ascii=False) + "\n")
            jf.flush()
            print(f"Segment {seg_idx} failed: {exc}")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

predictions_df = pd.DataFrame(results)
predictions_csv = OUTPUT_DIR / "qwen_3rs_predictions.csv"
predictions_df.to_csv(predictions_csv, index=False)

print("Saved:", predictions_csv)
print("Saved:", jsonl_path)
predictions_df.head()


## 11. Quick inspection plot

In [ ]:
import matplotlib.pyplot as plt

score_cols = ["WD_P_pred", "WD_T_pred", "CF_P_pred", "CF_T_pred"]

if not predictions_df.empty:
    plot_df = predictions_df.set_index("segment_idx")[score_cols]
    ax = plot_df.plot(marker="o", figsize=(12, 5))
    ax.set_xlabel("1-minute segment")
    ax.set_ylabel("Predicted 3RS salience")
    ax.set_ylim(0.8, 5.2)
    ax.set_title("Qwen2.5-VL-3B zero-shot rupture salience")
    plt.show()


## 12. Optional evaluation against human labels

Expected CSV columns:

```text
segment_idx,WD_P,WD_T,CF_P,CF_T
0,1.0,1.0,1.0,1.0
1,2.5,1.0,1.0,1.0
...
```

For PROTECT-AI these may be the mean ratings from two human coders.

The notebook reports MAE, exact agreement after rounding, within-one agreement, and exploratory binary rupture detection. The binary threshold is a secondary exploratory choice and should be prespecified before confirmatory analysis.


In [ ]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

def evaluate_against_humans(pred_df: pd.DataFrame, labels_csv: Path):
    human = pd.read_csv(labels_csv)
    merged = pred_df.merge(human, on="segment_idx", how="inner")

    metrics = []
    for target in ["WD_P", "WD_T", "CF_P", "CF_T"]:
        y_true = merged[target].astype(float).to_numpy()
        y_pred = merged[f"{target}_pred"].astype(float).to_numpy()

        metrics.append({
            "target": target,
            "n": len(y_true),
            "MAE": float(np.mean(np.abs(y_true - y_pred))),
            "exact_rounded": float(np.mean(np.rint(y_true) == np.rint(y_pred))),
            "within_one": float(np.mean(np.abs(y_true - y_pred) <= 1.0)),
        })

    metrics_df = pd.DataFrame(metrics)

    human_any = (
        merged[["WD_P", "WD_T", "CF_P", "CF_T"]].max(axis=1)
        >= BINARY_RUPTURE_THRESHOLD
    ).astype(int)

    pred_any = (
        merged[["WD_P_pred", "WD_T_pred", "CF_P_pred", "CF_T_pred"]].max(axis=1)
        >= BINARY_RUPTURE_THRESHOLD
    ).astype(int)

    binary_metrics = {
        "threshold": BINARY_RUPTURE_THRESHOLD,
        "balanced_accuracy": balanced_accuracy_score(human_any, pred_any),
        "precision": precision_score(human_any, pred_any, zero_division=0),
        "recall": recall_score(human_any, pred_any, zero_division=0),
        "f1": f1_score(human_any, pred_any, zero_division=0),
    }

    return merged, metrics_df, binary_metrics

if HUMAN_LABELS_CSV is not None:
    merged_df, ordinal_metrics_df, binary_metrics = evaluate_against_humans(
        predictions_df,
        Path(HUMAN_LABELS_CSV),
    )
    display(ordinal_metrics_df)
    print(binary_metrics)
else:
    print("Set HUMAN_LABELS_CSV when you are ready to compare with coder labels.")


## 13. Recommended interpretation of this first experiment

Treat this as a **zero-shot visual baseline**, not the final PROTECT-AI model.

Useful questions:

1. Can a general-purpose VLM identify any 3RS-relevant visible behavior without task-specific training?
2. Does it perform better for confrontation than withdrawal?
3. Does performance drop when the patient's or therapist's face is poorly visible?
4. Does the VLM over-predict rupture from ordinary gaze shifts, stillness, gestures or posture?
5. Do the explanations reference behavior that is actually visible?
6. How stable are results across video sampling rates?

A strong next comparison would be:

- **A:** VLM video only
- **B:** VLM video + engineered video features — this notebook
- **C:** transcript-only model
- **D:** audio + transcript
- **E:** full multimodal model

Use patient/session-level splits once you move from zero-shot exploration to trained evaluation.


## References used for the experiment design

- Eubanks, C. F., & Muran, J. C. — *Rupture Resolution Rating System (3RS): Manual Version 2022*.
- Qwen Team — *Qwen2.5-VL-3B-Instruct* model documentation/model card.
- PROTECT-AI current annotation setup — one-minute segments with separate 1–5 ratings for patient withdrawal, therapist withdrawal, patient confrontation and therapist confrontation.
